In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
from langchain.vectorstores import LanceDB
import lancedb
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import PyPDFLoader

In [3]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/var/folders/mc/13pzbbl540bf1lf63sh1nd_w0000gn/T/ipykernel_60444/323160131.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# Load and split PDF
loader = PyPDFLoader("design patterns.pdf")
pages = loader.load_and_split()

In [5]:
# BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(pages)
bm25_retriever.k = 2  # Retrieve top 2 results

In [8]:
# Connect to LanceDB
db = lancedb.connect('/tmp/lancedb')  
table = db.create_table("pandas_docs", data=[{"vector": embedding.embed_query("Hello World"), "text": "Hello World", "id": "1"}], mode="overwrite")

# Initialize LanceDB retriever
docsearch = LanceDB.from_documents(pages, embedding, connection=db)
retriever_lancedb = docsearch.as_retriever(search_kwargs={"k": 2})

In [17]:
# Ensemble Retriever combining BM25 & LanceDB
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, retriever_lancedb], weights=[0.4, 0.6])

# Example query
query = "What is a singleton design pattern?"

# Retrieve relevant documents
docs = ensemble_retriever.get_relevant_documents(query)

# Print results
for doc in docs:
    print('__________________________')
    print(doc.page_content)


__________________________
In order to implement a singleton design pattern, the bestpractice is to build the “one and only one” goal into theclassitself so that creating another instance of a Singleton class isnot even possible. This “codifies” the design intent within thesoftware. This is necessary if workingonalargeproject or onprojects with multiple developers, but it is helpful even onsmaller or individual projects.
Let us examine this in code.
If aclasshasapublicconstructor,anobjectofthisclasscanbeinstantiated at any time.
publicclassnotSingleton {//Public ConstructorpublicnotSingleton() {..}}
Instead, if the class has a private constructor, then theconstructor cannot be called fromoutside the class. Althoughthis seems toprevent creatinganobject of theclass, therearetwo key components for getting around this.
First, declare a class variable. In this case, it is called “uniqueInstance.” This class variable will refer to the one instance ofyour Singleton class. As the variable is p